<a href="https://colab.research.google.com/github/aravindanmoorthy/Claude-Hackathon/blob/claude%2Fweather-alerts-parked-cars-58mry/Copy_of_safe_pilot_weather_alert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Safe Pilot — Weather Alert System
### USAA Hackathon Project

Checks real-time weather for GPS locations and determines whether a **Safe Pilot alert** should be triggered for a **parked vehicle**.

**API:** [Open-Meteo](https://open-meteo.com/) — Free, no API key required!

---
**This notebook has two sections:**
- 🌍 **Section A** — Real live weather for 5 locations (results vary by current conditions)
- 🧪 **Section B** — Simulated scenarios with forced weather codes (always shows alerts for demo)

> Run each cell top to bottom using **Shift+Enter**

In [ ]:
!pip install requests --quiet
print('✅ Dependencies ready!')

In [ ]:
import requests
from dataclasses import dataclass
from datetime import datetime
from copy import deepcopy

print('✅ Imports successful!')

In [ ]:
ALERT_THRESHOLD_CODES = {95, 96, 99, 82, 75, 65, 45, 48}

SEVERE_WEATHER_CODES = {
    0:  'Clear Sky',
    1:  'Mainly Clear',
    2:  'Partly Cloudy',
    3:  'Overcast',
    45: 'Foggy',
    48: 'Icy Fog',
    51: 'Light Drizzle',
    61: 'Light Rain',
    63: 'Moderate Rain',
    65: 'Heavy Rain',
    71: 'Light Snow',
    73: 'Moderate Snow',
    75: 'Heavy Snow',
    77: 'Snow Grains',
    80: 'Rain Showers',
    81: 'Heavy Showers',
    82: 'Violent Showers',
    85: 'Snow Showers',
    95: 'Thunderstorm',
    96: 'Thunderstorm + Hail',
    99: 'Thunderstorm + Heavy Hail',
}

ALERT_SEVERITY = {
    45: 'MEDIUM',
    48: 'HIGH',
    65: 'MEDIUM',
    75: 'HIGH',
    82: 'HIGH',
    95: 'HIGH',
    96: 'HIGH',
    99: 'CRITICAL',
}

# Specific risk advice per condition for parked vehicles
PARKED_VEHICLE_ADVICE = {
    45: '🌫️  Move to a covered garage — low visibility risk for other drivers hitting your parked car.',
    48: '🌫️  Icy fog risk. Move vehicle to a covered area to prevent ice buildup on windshield.',
    65: '🌧️  Heavy rain expected. Move away from low-lying areas to avoid flood risk.',
    75: '❄️   Heavy snow warning! Move vehicle to a garage to prevent snow/ice damage.',
    82: '🌊  Violent rain showers. Seek covered parking immediately — flash flood risk.',
    95: '⛈️   Thunderstorm approaching! Move vehicle away from trees and open fields.',
    96: '⛈️   Thunderstorm with HAIL! Move to covered parking NOW to prevent hail damage.',
    99: '🚨  CRITICAL: Thunderstorm with HEAVY HAIL! Immediate covered shelter required — severe vehicle damage risk.',
}

print('✅ Constants loaded!')

In [ ]:
@dataclass
class WeatherAlert:
    location_name:    str
    lat:              float
    lon:              float
    is_severe_now:    bool
    condition_now:    str
    weather_code:     int
    wind_speed_mph:   float
    visibility_miles: float
    temperature_f:    float
    upcoming_alerts:  list
    alert_required:   bool
    severity:         str
    alert_message:    str
    vehicle_advice:   str

print('✅ WeatherAlert data class defined!')

In [ ]:
def fetch_weather(lat: float, lon: float) -> dict:
    """Call Open-Meteo API with lat/lon. Returns raw JSON."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'current': ['temperature_2m', 'wind_speed_10m', 'weather_code', 'precipitation', 'visibility'],
        'hourly':  ['precipitation_probability', 'wind_gusts_10m', 'weather_code', 'visibility'],
        'temperature_unit': 'fahrenheit',
        'wind_speed_unit':  'mph',
        'forecast_days': 1
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException as e:
        print(f'  API call failed: {e}')
        return None

print('✅ fetch_weather() defined!')

In [ ]:
def parse_weather_alert(location: dict, response: dict) -> WeatherAlert:
    """Parse raw API response into a WeatherAlert object."""
    current = response['current']
    hourly  = response['hourly']

    weather_code     = current['weather_code']
    wind_speed_mph   = current['wind_speed_10m']
    visibility_mi    = current['visibility'] / 1609
    temperature_f    = current['temperature_2m']
    condition_now    = SEVERE_WEATHER_CODES.get(weather_code, 'Unknown')
    is_severe_now    = weather_code in ALERT_THRESHOLD_CODES

    upcoming_alerts = []
    for i, code in enumerate(hourly['weather_code'][:1]):
        if code in ALERT_THRESHOLD_CODES:
            upcoming_alerts.append({
                'hour':       hourly['time'][i],
                'condition':  SEVERE_WEATHER_CODES.get(code, 'Unknown'),
                'rain_prob':  hourly['precipitation_probability'][i],
                'wind_gusts': hourly['wind_gusts_10m'][i],
            })

    alert_required = is_severe_now or len(upcoming_alerts) > 0

    severity = 'NONE'
    if is_severe_now:
        severity = ALERT_SEVERITY.get(weather_code, 'MEDIUM')
    elif upcoming_alerts:
        severity = ALERT_SEVERITY.get(hourly['weather_code'][0], 'MEDIUM')

    vehicle_advice = PARKED_VEHICLE_ADVICE.get(weather_code, '')

    if alert_required:
        if is_severe_now:
            alert_message = (
                f'SAFE PILOT ALERT [{severity}]: {condition_now} detected! '
                f'Wind: {wind_speed_mph:.1f} mph | Visibility: {visibility_mi:.1f} miles.'
            )
        else:
            u = upcoming_alerts[0]
            alert_message = (
                f"SAFE PILOT ALERT [{severity}]: {u['condition']} expected within 1 hour "
                f"(Rain: {u['rain_prob']}% | Gusts: {u['wind_gusts']:.1f} mph)."
            )
    else:
        alert_message = 'No alert needed. Weather conditions are safe.'

    return WeatherAlert(
        location_name=location['name'], lat=location['lat'], lon=location['lon'],
        is_severe_now=is_severe_now, condition_now=condition_now,
        weather_code=weather_code, wind_speed_mph=wind_speed_mph,
        visibility_miles=visibility_mi, temperature_f=temperature_f,
        upcoming_alerts=upcoming_alerts, alert_required=alert_required,
        severity=severity, alert_message=alert_message, vehicle_advice=vehicle_advice,
    )

print('✅ parse_weather_alert() defined!')

In [ ]:
def print_alert(alert: WeatherAlert):
    print('=' * 65)
    print(f'  📍 {alert.location_name}')
    print(f'     Lat: {alert.lat} | Lon: {alert.lon}')
    print('-' * 65)
    print(f'  🌡️  Temperature   : {alert.temperature_f:.1f} F')
    print(f'  🌤️  Condition Now : {alert.condition_now} (code {alert.weather_code})')
    print(f'  💨  Wind Speed    : {alert.wind_speed_mph:.1f} mph')
    print(f'  👁️  Visibility    : {alert.visibility_miles:.1f} miles')
    if alert.upcoming_alerts:
        print('\n  📅 Upcoming (next 1 hour):')
        for a in alert.upcoming_alerts:
            print(f"     • {a['hour']} → {a['condition']} | Rain: {a['rain_prob']}% | Gusts: {a['wind_gusts']:.1f} mph")
    if alert.alert_required:
        print(f'\n  🚨 {alert.alert_message}')
        if alert.vehicle_advice:
            print(f'  💡 Advice: {alert.vehicle_advice}')
    else:
        print(f'\n  ✅ {alert.alert_message}')
    print('=' * 65)
    print()

print('✅ print_alert() defined!')

In [ ]:
def check_weather_at_location(location: dict):
    """Full pipeline: lat/lon → API → parse → alert."""
    print(f"\n🔍 Checking: {location['name']} ...")
    response = fetch_weather(location['lat'], location['lon'])
    if response is None:
        print('  ❌ Could not retrieve weather data.')
        return None
    alert = parse_weather_alert(location, response)
    print_alert(alert)
    return alert

print('✅ check_weather_at_location() defined!')

---
## 🌍 Section A — Real Live Weather (5 Locations)
Results depend on actual current weather conditions.

In [ ]:
REAL_LOCATIONS = [
    {'name': 'Edinburg, TX',         'lat': 26.3017,  'lon': -98.1633},
    {'name': 'Miami, FL',            'lat': 25.7617,  'lon': -80.1918},
    {'name': 'Denver, CO',           'lat': 39.7392,  'lon': -104.9903},
    {'name': 'Oklahoma City, OK',    'lat': 35.4676,  'lon': -97.5164},
    {'name': 'San Diego, CA',        'lat': 32.7157,  'lon': -117.1611},
]

print('\n' + '🛡️  SAFE PILOT — SECTION A: Live Weather'.center(65))
print(f"{'Run at: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'):^65}\n")

alerts_triggered, no_alerts = [], []
for location in REAL_LOCATIONS:
    alert = check_weather_at_location(location)
    if alert:
        (alerts_triggered if alert.alert_required else no_alerts).append(alert.location_name)

print('\n' + '-' * 65)
print('📊 SECTION A SUMMARY')
print('-' * 65)
if alerts_triggered:
    print(f'  🚨 Alerts Triggered ({len(alerts_triggered)}):')
    for name in alerts_triggered: print(f'     • {name}')
if no_alerts:
    print(f'  ✅ No Alert Needed ({len(no_alerts)}):')
    for name in no_alerts: print(f'     • {name}')
print('-' * 65)

---
## 🧪 Section B — Simulated Alert Scenarios (Parked Vehicle)

These scenarios **inject specific weather codes** into a real API response to simulate
conditions that a parked vehicle owner must act on. Useful for demos and testing.

| Scenario | Weather Code | Condition | Severity |
|---|---|---|---|
| 1 | 99 | Thunderstorm + Heavy Hail | 🔴 CRITICAL |
| 2 | 96 | Thunderstorm + Hail | 🔴 HIGH |
| 3 | 95 | Thunderstorm | 🔴 HIGH |
| 4 | 82 | Violent Rain Showers | 🟠 HIGH |
| 5 | 75 | Heavy Snow | 🟠 HIGH |
| 6 | 48 | Icy Fog | 🟡 MEDIUM |
| 7 | 65 | Heavy Rain | 🟡 MEDIUM |
| 8 | 0  | Clear Sky | ✅ NONE |

In [ ]:
def build_mock_response(base_response: dict, weather_code: int,
                         wind_mph: float = 0.0, visibility_m: float = 24000.0,
                         rain_prob: int = 0, wind_gusts: float = 0.0) -> dict:
    """
    Clone a real API response and inject a specific weather code + conditions.
    This lets us simulate any alert scenario without needing real storms.
    """
    mock = deepcopy(base_response)
    mock['current']['weather_code']   = weather_code
    mock['current']['wind_speed_10m'] = wind_mph
    mock['current']['visibility']     = visibility_m
    # Inject into first hourly slot too
    mock['hourly']['weather_code'][0]              = weather_code
    mock['hourly']['precipitation_probability'][0] = rain_prob
    mock['hourly']['wind_gusts_10m'][0]            = wind_gusts
    return mock

print('✅ build_mock_response() defined!')

In [ ]:
# Use Edinburg TX as base location for all simulated scenarios
BASE_LOCATION = {'name': 'Edinburg, TX', 'lat': 26.3017, 'lon': -98.1633}

print('Fetching base weather response for simulation...')
base_response = fetch_weather(BASE_LOCATION['lat'], BASE_LOCATION['lon'])

if base_response is None:
    print('Could not fetch base response. Check internet connection.')
else:
    SIMULATED_SCENARIOS = [
        {
            'location': {'name': 'Scenario 1 — Thunderstorm + Heavy Hail (CRITICAL)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 99, 'wind_mph': 68.0, 'visibility_m': 800,
            'rain_prob': 95, 'wind_gusts': 82.0
        },
        {
            'location': {'name': 'Scenario 2 — Thunderstorm + Hail (HIGH)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 96, 'wind_mph': 52.0, 'visibility_m': 1200,
            'rain_prob': 88, 'wind_gusts': 65.0
        },
        {
            'location': {'name': 'Scenario 3 — Thunderstorm (HIGH)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 95, 'wind_mph': 43.0, 'visibility_m': 2500,
            'rain_prob': 80, 'wind_gusts': 58.0
        },
        {
            'location': {'name': 'Scenario 4 — Violent Rain Showers (HIGH)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 82, 'wind_mph': 38.0, 'visibility_m': 3000,
            'rain_prob': 75, 'wind_gusts': 47.0
        },
        {
            'location': {'name': 'Scenario 5 — Heavy Snow (HIGH)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 75, 'wind_mph': 25.0, 'visibility_m': 500,
            'rain_prob': 90, 'wind_gusts': 35.0
        },
        {
            'location': {'name': 'Scenario 6 — Icy Fog (MEDIUM)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 48, 'wind_mph': 8.0, 'visibility_m': 200,
            'rain_prob': 30, 'wind_gusts': 12.0
        },
        {
            'location': {'name': 'Scenario 7 — Heavy Rain (MEDIUM)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 65, 'wind_mph': 20.0, 'visibility_m': 4000,
            'rain_prob': 65, 'wind_gusts': 28.0
        },
        {
            'location': {'name': 'Scenario 8 — Clear Sky (NO ALERT)',
                         'lat': 26.3017, 'lon': -98.1633},
            'code': 0, 'wind_mph': 5.0, 'visibility_m': 24000,
            'rain_prob': 0, 'wind_gusts': 7.0
        },
    ]

    print('\n' + '🧪  SAFE PILOT — SECTION B: Simulated Scenarios'.center(65) + '\n')

    alerts_triggered, no_alerts = [], []

    for s in SIMULATED_SCENARIOS:
        mock = build_mock_response(
            base_response, s['code'],
            wind_mph=s['wind_mph'], visibility_m=s['visibility_m'],
            rain_prob=s['rain_prob'], wind_gusts=s['wind_gusts']
        )
        print(f"\n🔍 Running: {s['location']['name']}")
        alert = parse_weather_alert(s['location'], mock)
        print_alert(alert)
        (alerts_triggered if alert.alert_required else no_alerts).append(alert.location_name)

    print('\n' + '-' * 65)
    print('📊 SECTION B SUMMARY')
    print('-' * 65)
    if alerts_triggered:
        print(f'  🚨 Alerts Triggered ({len(alerts_triggered)}):')
        for name in alerts_triggered: print(f'     • {name}')
    if no_alerts:
        print(f'  ✅ No Alert Needed ({len(no_alerts)}):')
        for name in no_alerts: print(f'     • {name}')
    print('-' * 65)

---
## ✏️ Section C — Test Your Own Lat/Lon

In [ ]:
custom_location = {
    'name': 'My Custom Location',
    'lat':  26.3017,   # Replace with your latitude
    'lon': -98.1633    # Replace with your longitude
}

check_weather_at_location(custom_location)